# Automated Model with Functions and stuff

### convenevoli iniziali

In [1]:
!pip install xlsxwriter

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)

### upload of the zip file

In [3]:
from google.colab import files
uploaded = files.upload()

Saving paridis2425.zip to paridis2425 (1).zip


## Individual functions

### Unzipper function

In [4]:
import zipfile
import os

def extract_uploaded_data(zip_file_path, extract_to_folder='/content/raw_data'):
    """
    Unzips the uploaded file and flattens the directory structure completely,
    ensuring every single file inside any subfolder is extracted directly
    into the root destination folder.
    """
    print(f"📦 Extracting {zip_file_path}...")

    # Create the target directory if it doesn't exist
    os.makedirs(extract_to_folder, exist_ok=True)

    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # Loop through every single item inside the ZIP archive
        for member in zip_ref.infolist():
            # Skip directory entries themselves (we only care about actual files)
            if member.is_dir():
                continue

            # Get just the filename, ignoring the internal folder path
            filename = os.path.basename(member.filename)

            # Skip system files like macOS __MACOSX hidden folders or .DS_Store
            if not filename or filename.startswith('._') or filename == '.DS_Store':
                continue

            # Define the final destination path directly in the root folder
            target_path = os.path.join(extract_to_folder, filename)

            # --- Optional: Handle Name Collisions ---
            # If two different subfolders have a file with the exact same name,
            # this prevents them from overwriting each other.
            base, ext = os.path.splitext(filename)
            counter = 1
            while os.path.exists(target_path):
                target_path = os.path.join(extract_to_folder, f"{base}_{counter}{ext}")
                counter += 1
            # ----------------------------------------

            # Read the file data from the ZIP and write it to the flat destination
            with zip_ref.open(member) as source_file, open(target_path, "wb") as dest_file:
                dest_file.write(source_file.read())

    print(f"✅ Extraction complete. All files are flattened directly into: {extract_to_folder}")
    return extract_to_folder

### Year-Week converter

In [5]:
def year_week_converter(df,day_column='Giorno'):
    # Create unique string year-week ISO (es. '2023-W52', '2024-W01')
    # %G = year ISO, %V = week nr. ISO (01-53)
    df['Anno_Settimana'] = df[day_column].dt.strftime('%G-W%V')

    # move it to 3rd place
    col_settimana = df.pop('Anno_Settimana')
    df.insert(2, 'Anno_Settimana', col_settimana)

    return df

### File reader, categorizer, and basic cleaning function

In [6]:
import os
import glob
import pandas as pd

def process_raw_network_data(folder_path):
    """
    Loops through a folder of raw Excel files, categorizes them based on metadata,
    extracts variables, cleans the headers, and routes them to the correct DataFrame.
    """
    print(f"📂 Scanning folder: {folder_path}")

    punct_dfs = []
    supp_tot_dfs = []
    supp_parz_dfs = []

    # 1. Look for Excel files instead of CSVs
    all_files = glob.glob(os.path.join(folder_path, '*.xlsx'))

    for file in all_files:
        try:
            # 2. Read metadata using read_excel.
            # usecols=[0] speeds it up since metadata flags are always in Column A
            meta_df = pd.read_excel(file, header=None, nrows=30, usecols=[0])

            first_cell = str(meta_df.iloc[0, 0]).strip()
            meta_text = ' '.join(meta_df[0].dropna().astype(str))

            # Find the actual Header Row index (Anchor on 'VISTE')
            header_idx = meta_df[meta_df[0].astype(str).str.upper() == 'VISTE'].index[0]

            # ---------------------------------------------------------
            # CATEGORY A: PUNCTUALITY
            # ---------------------------------------------------------
            if first_cell.startswith("COP"):
                if "Verso = 'Dispari'" in meta_text:
                    direction = 1
                elif "Verso = 'Pari'" in meta_text:
                    direction = 2
                else:
                    direction = None

                # 3. Load full table with read_excel
                df = pd.read_excel(file, skiprows=header_idx)
                # skip first row because it's a summary
                df = df.iloc[1:].reset_index(drop=True)

                df['Direction'] = direction
                col_name = 'Viste' if 'Viste' in df.columns else 'VISTE'
                df['Short_Viste'] = df[col_name].astype(str).str.split('_').str[0]

                punct_dfs.append(df)

            # ---------------------------------------------------------
            # CATEGORY B: SUPPRESSIONS
            # ---------------------------------------------------------
            elif first_cell.startswith("Volumi"):
                gruppo_row = meta_df[meta_df[0].astype(str).str.contains("Gruppo =", na=False)]
                gruppo_str = gruppo_row.iloc[0, 0].replace("Gruppo =", "").replace("'", "").strip()
                short_vista = gruppo_str.split('_')[0]

                # 3. Load full table with read_excel
                df = pd.read_excel(file, skiprows=header_idx)
                df = df.iloc[1:].reset_index(drop=True)

                df['VISTE'] = gruppo_str
                df['Short_Viste'] = short_vista

                if "Soppressione Totale" in meta_text:
                    supp_tot_dfs.append(df)
                else:
                    supp_parz_dfs.append(df)

            else:
                print(f"⚠️ Warning: Could not categorize {os.path.basename(file)}")

        except Exception as e:
            print(f"❌ Error reading file {os.path.basename(file)}: {e}")

    # ---------------------------------------------------------
    # FINAL ASSEMBLY & MERGING
    # ---------------------------------------------------------
    df_punct_final = pd.concat(punct_dfs, ignore_index=True) if punct_dfs else pd.DataFrame()
    if not df_punct_final.empty:
        # Move the Short_Viste column
        short_viste = df_punct_final.pop('Short_Viste')
        df_punct_final.insert(0, 'Short_Viste', short_viste)

    df_supp_tot = pd.concat(supp_tot_dfs, ignore_index=True) if supp_tot_dfs else pd.DataFrame()
    df_supp_parz = pd.concat(supp_parz_dfs, ignore_index=True) if supp_parz_dfs else pd.DataFrame()

    df_supp_final = pd.DataFrame()

    if not df_supp_tot.empty and not df_supp_parz.empty:
        print("🔗 Merging Total and Partial Suppressions...")
        columns_to_drop = ['VISTE', 'Mese', 'Anno']
        cols_to_drop_safe = [c for c in columns_to_drop if c in df_supp_parz.columns]

        # 1. Do an OUTER merge with indicator=True
        temp_merge = pd.merge(
            df_supp_tot,
            df_supp_parz.drop(columns=cols_to_drop_safe),
            on=['Short_Viste', 'Giorno'],
            suffixes=('_T', '_P'),
            how='outer',
            indicator=True  # Tracks where the data came from
        )

        # 2. Count the mismatched rows
        missing_partial = (temp_merge['_merge'] == 'left_only').sum()
        missing_total = (temp_merge['_merge'] == 'right_only').sum()
        total_dropped = missing_partial + missing_total

        # 3. Print the diagnostic report
        if total_dropped > 0:
            print(f"⚠️ Inner Join Dropped {total_dropped} rows:")
            print(f"   - {missing_partial} rows had Total suppressions but NO Partial suppressions.")
            print(f"   - {missing_total} rows had Partial suppressions but NO Total suppressions.")

        # 4. Enforce the "inner" logic by keeping only rows present in "both"
        df_supp_final = temp_merge[temp_merge['_merge'] == 'both'].copy()
        df_supp_final = df_supp_final.drop(columns=['_merge'])

        short_viste = df_supp_final.pop('Short_Viste')
        df_supp_final.insert(0, 'Short_Viste', short_viste)

    elif not df_supp_tot.empty:
        df_supp_final = df_supp_tot
    elif not df_supp_parz.empty:
        df_supp_final = df_supp_parz

    print(f"✅ Done! Ready: Punctuality rows: {len(df_punct_final)}, Suppressions rows: {len(df_supp_final)}")

    # add year_week column
    if not df_punct_final.empty:
        df_punct_final = year_week_converter(df_punct_final)
    if not df_supp_final.empty:
        df_supp_final = year_week_converter(df_supp_final)

    return df_punct_final, df_supp_final

### Data integrity checks

In [7]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. THE QUARANTINE ENGINE FUNCTION
# ---------------------------------------------------------
def run_sanity_checks(df, rules_dict, columns_to_keep=None):
    """
    Evaluates rules against a DataFrame. Flags failures, prints a report,
    purges bad rows, and TRIMS the DataFrame to keep only specified columns.
    """
    if df.empty:
        return df

    print(f"🕵️ Running Sanity Checks (Initial rows: {len(df)})")

    bad_indices = set()
    report = []

    # 1. Evaluate each rule
    for rule_name, failure_condition in rules_dict.items():
        try:
            failed_mask = failure_condition(df).fillna(False)
            failed_count = failed_mask.sum()

            if failed_count > 0:
                bad_indices.update(df[failed_mask].index)

            report.append({'Rule': rule_name, 'Failed Rows': failed_count})
        except Exception as e:
            report.append({'Rule': rule_name, 'Failed Rows': f"Error: {e}"})

    # 2. Print the Diagnostic Report
    report_df = pd.DataFrame(report)
    print(report_df.to_string(index=False))

    # 3. Purge the bad rows
    clean_df = df.drop(index=list(bad_indices)).reset_index(drop=True)
    print(f"🧹 Purged {len(bad_indices)} total invalid rows.")

    # Change the names of the columns
    clean_df = clean_df.rename(columns={
    'Treni con Traccia Programmata_P': 'Media_Programmati_Supp',
    'Treni Circolati_P': 'Media_Circolati_Supp'
    })

    # 4. Strict Column Subsetting & Warning System
    if columns_to_keep:
        # Check for missing columns
        missing_cols = [c for c in columns_to_keep if c not in clean_df.columns]
        if missing_cols:
            print(f"🚨 WARNING: The following requested columns were NOT FOUND:")
            print(f"   -> {missing_cols}")
            print("   -> Proceeding subset with available columns only.")

        # Only keep the requested columns that actually exist
        cols_present = [c for c in columns_to_keep if c in clean_df.columns]
        clean_df = clean_df[cols_present]
        print(f"✂️ Trimmed down to {len(cols_present)} core columns.")

    print(f"✅ Final rows: {len(clean_df)}\n" + "-"*40)

    return clean_df


### Aggregation of daily data into weekly

In [8]:
import pandas as pd
import numpy as np

def aggregate_data_weekly(df, group_cols):
    """
    Groups data to a weekly level. Automatically sums numeric columns,
    counts unique days, explicitly averages 'Media Treni Circolati' if present,
    and logs exactly how many incomplete weeks were filtered out.
    """
    print(f"🔄 Aggregating data to Weekly by {group_cols}...")

    # 1. Base rule: count unique days
    agg_rules = {'Giorno': 'nunique'}

    # 2. Find all numeric columns
    numeric_cols = df.select_dtypes(include='number').columns

    # 3. Assign rules dynamically
    for col in numeric_cols:
        if col not in group_cols:
            if col == 'Media Treni Circolati':
                agg_rules[col] = 'mean'
            elif col == 'Media_Programmati_Supp':
                agg_rules[col] = 'mean'
            elif col == 'Media_Circolati_Supp':
                agg_rules[col] = 'mean'
            else:
                agg_rules[col] = 'sum'

    # 4. Group and Aggregate
    df_weekly = df.groupby(group_cols, as_index=False).agg(agg_rules)

    # 5. NEW: Track and Filter for valid weeks (5+ days)
    initial_weeks = len(df_weekly)
    df_weekly = df_weekly[df_weekly['Giorno'] >= 5].copy()

    dropped_weeks = initial_weeks - len(df_weekly)
    if dropped_weeks > 0:
        print(f"   ⚠️ Dropped {dropped_weeks} incomplete weeks (< 5 active days).")

    df_weekly.rename(columns={'Giorno': 'Giorni_Registrati'}, inplace=True)

    print(f"✅ Aggregation complete. Final Rows: {len(df_weekly)}\n" + "-"*40)
    return df_weekly.reset_index(drop=True)

### Defective routes and Small weeks filtration

In [9]:
def filter_weekly(df, avg_trains_column, routes_to_drop=None, min_trains_per_day=3):
    """
    Removes arbitrarily excluded routes, routes with a numeric prefix > 30 (or exactly 0) as they don't exist,
    and weeks with very low average circulation. Provides granular reporting on dropped rows.
    """
    if routes_to_drop is None:
        routes_to_drop = ['23. MI1', '24. MI2', '25. MI2', '26. MI1']

    print(f"🧹 Applying Post-Aggregation Filters to Punctuality...")
    initial_rows = len(df)

    # 1. Identify Bad Routes (Arbitrary & Prefixes)
    route_prefix_nums = df['Short_Viste'].str.extract(r'^(\d+)')[0].astype(float).fillna(-1)
    bad_prefix_mask = (route_prefix_nums > 30) | (route_prefix_nums == 0)
    arbitrary_mask = df['Short_Viste'].isin(routes_to_drop)

    # Apply Step 1: Drop Bad Routes
    df_step1 = df[~bad_prefix_mask & ~arbitrary_mask].copy()
    dropped_routes = initial_rows - len(df_step1)

    # Apply Step 2: Drop Low Circulation
    df_filtered = df_step1[df_step1[avg_trains_column] > min_trains_per_day].copy()
    dropped_low_circ = len(df_step1) - len(df_filtered)

    # Print the granular report
    if dropped_routes > 0:
        print(f"   ⚠️ Dropped {dropped_routes} rows(weeks) (Excluded routes or prefix >30/00).")
    if dropped_low_circ > 0:
        print(f"   ⚠️ Dropped {dropped_low_circ} rows(weeks) (<= {min_trains_per_day} average trains/day).")

    print(f"✅ Filter complete. Ready for Pivot. Final Rows: {len(df_filtered)}\n" + "-"*40)

    return df_filtered.reset_index(drop=True)

### Group directions and calculate directional asymmetries in punctuality and operated trains (Punctuality only)

In [10]:
import pandas as pd
import numpy as np

def pivot_and_group_directions(df):
    # 1. Calculate temporary directional punctuality
    df['Temp_Punt'] = (df['In Fascia'] / df['Circolati'].replace(0, np.nan)) * 100

    # Pivot BOTH Punctuality and Circolati
    df_pivot = df.pivot(
        index=['Short_Viste', 'Anno_Settimana'],
        columns='Direction',
        values=['Temp_Punt', 'Circolati']
    )

    # Flatten the column names (creates 'Temp_Punt_1', 'Circolati_2', etc.)
    df_pivot.columns = [f"{col[0]}_{col[1]}" for col in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # Safely extract directional circulation
    c1 = df_pivot.get('Circolati_1', np.nan)
    c2 = df_pivot.get('Circolati_2', np.nan)

    # Calculate Differences
    df_pivot['Diff_True_Punct'] = df_pivot.get('Temp_Punt_1', np.nan) - df_pivot.get('Temp_Punt_2', np.nan)

    # NORMALIZED Circolati Difference: (Dir 1 - Dir 2) / Total * 100
    # Note: c1 + c2 will naturally result in NaN if a direction is missing, keeping the math safe.
    df_pivot['Diff_Circolati_Norm'] = ((c1 - c2) / (c1 + c2).replace(0, np.nan)) * 100

    df_asym = df_pivot[['Short_Viste', 'Anno_Settimana', 'Diff_True_Punct', 'Diff_Circolati_Norm']]

    # 2. Aggregate network volumes
    group_cols = ['Short_Viste', 'Viste', 'Anno_Settimana']

    agg_rules = {
        col: 'mean' if col == 'Giorni_Registrati' else 'sum'
        for col in df.select_dtypes(include='number').columns
        if col not in group_cols and col not in ['Direction', 'Temp_Punt']
    }

    df_network = df.groupby(group_cols, as_index=False).agg(agg_rules)

    # 3. Merge Asymmetries back in
    df_final = pd.merge(df_network, df_asym, on=['Short_Viste', 'Anno_Settimana'], how='left')

    # 4. Report single-direction weeks
    missing_asym = df_final['Diff_True_Punct'].isna().sum()
    if missing_asym > 0:
        print(f"   ⚠️ Weeks with only one direction (Asymmetry = NaN): {missing_asym}")

    return df_final

### Create final df with both punctuality and suppressions

In [11]:
import pandas as pd

def create_master_dataset(df_punct=None, df_supp=None, merge_type='inner'):
    """
    Merges Punctuality and Suppressions tables.
    Gracefully handles missing datasets and allows dynamic merge types.
    """
    print(f"🔄 Creating Master Dataset (Merge type: {merge_type})...")

    # Scenario 1: Only Punctuality is available
    if df_supp is None or df_supp.empty:
        print("⚠️ Only Punctuality data detected. Returning Punctuality without merging.")
        return df_punct.copy() if df_punct is not None else None

    # Scenario 2: Only Suppressions is available
    if df_punct is None or df_punct.empty:
        print("⚠️ Only Suppressions data detected. Returning Suppressions without merging.")
        return df_supp.copy()

    # Scenario 3: Both datasets exist -> Perform the merge
    if 'VISTE' in df_supp.columns:
        df_supp = df_supp.drop(columns=['VISTE'])

    df_master = pd.merge(
        df_punct,
        df_supp,
        on=['Short_Viste', 'Anno_Settimana'],
        how=merge_type,
        suffixes=('', '_Supp')
    )

    # Optional clean-up for the overlapping day counts
    if 'Giorni_Registrati_Supp' in df_master.columns:
        df_master = df_master.rename(columns={'Giorni_Registrati': 'Giorni_Registrati_Punct'})

    print(f"Final Master Dataset complete. Total Rows: {len(df_master)}")

    return df_master

### Calculate final metrics

In [12]:
import numpy as np

def calculate_final_metrics(df):
    """
    Calculates final network percentages for both Punctuality and Suppressions
    on the merged master dataset.
    """
    print("🧮 Calculating final metrics (Punctuality & Suppressions)...")
    df_out = df.copy()

    # Safely handle zeros in circulation
    circ = df_out['Circolati'].replace(0, np.nan)

    # 1. Calculate Punctuality Metrics
    if 'In Fascia' in df_out.columns:
        df_out['Punt. Reale'] = (df_out['In Fascia'] / circ) * 100
        df_out['Punt. RFI'] = ((df_out['Circolati'] - df_out['Fuori Fascia per RFI']) / circ) * 100
        df_out['Punt. IF'] = ((df_out['Circolati'] - df_out['Fuori Fascia per Propria IF']) / circ) * 100

    # 2. Calculate Suppression Metrics
    if 'Treni Soppressi_T' in df_out.columns:
        scheduled = df_out['Treni con Traccia Programmata_T']
        scheduled_safe = scheduled.replace(0, np.nan)

        df_out['Tasso_Soppressioni_Totali'] = (df_out['Treni Soppressi_T'] / scheduled_safe) * 100

        if 'Treni Soppressi_P' in df_out.columns:
            df_out['Tasso_Soppressioni_Parziali'] = (df_out['Treni Soppressi_P'] / scheduled_safe) * 100

    return df_out

## Analytical Functions

### Calculate historical trends and thresholds

In [13]:
import numpy as np

def calculate_historical_trends(df, metrics_config, windows):
    """
    Calculates historical rolling averages, deltas, and alert statuses.
    metrics_config: dict format {'ColumnName': ('alert_direction', threshold)}
    """
    print(f"📈 Calculating historical trends for windows: {windows}...")
    df_out = df.copy()

    # Sort chronologically for accurate rolling math
    df_out = df_out.sort_values(by=['Short_Viste', 'Anno_Settimana']).reset_index(drop=True)
    grouped = df_out.groupby('Short_Viste')

    for w in windows:
        for metric, (alert_direction, threshold) in metrics_config.items():
            if metric not in df_out.columns:
                continue

            # 1. STRICT WINDOWING: Removed min_periods=1
            # It now requires exactly 'w' previous weeks to calculate an average.
            hist_avg = grouped[metric].transform(lambda x: x.rolling(window=w).mean().shift(1))

            ma_col = f'RollingAvg_{metric}_{w}w'
            delta_col = f'Delta_{metric}_{w}w'
            status_col = f'Status_{metric}_{w}w'

            df_out[ma_col] = hist_avg
            df_out[delta_col] = df_out[metric] - hist_avg

            # 2. SAFE THRESHOLDS: If Delta is NaN, leave Status as NaN instead of 0
            if alert_direction == 'higher_is_bad':
                df_out[status_col] = np.where(
                    df_out[delta_col].isna(),
                    np.nan,
                    np.where(df_out[delta_col] > threshold, 1, 0)
                )
            elif alert_direction == 'lower_is_bad':
                df_out[status_col] = np.where(
                    df_out[delta_col].isna(),
                    np.nan,
                    np.where(df_out[delta_col] < -abs(threshold), 1, 0)
                )

            elif alert_direction == 'either_is_bad':
                df_out[status_col] = np.where(
                    df_out[delta_col].isna(),
                    np.nan,
                    np.where(abs(df_out[delta_col]) > abs(threshold), 1, 0)
                )

    return df_out

### Produce executive report

In [14]:
import pandas as pd
import numpy as np
import xlsxwriter

def generate_executive_report(df, trend_config, window=4, output_filename='Executive_Report.xlsx'):
    """
    Generates a multi-tabbed Excel report automatically mapping to the
    columns generated by 'calculate_historical_trends', including thresholds and all-time percentiles.
    """
    # 1. Pre-processing
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.sort_values(['Viste', 'Anno_Settimana'])
    df['Relazione']=df['Viste'].astype(str).str.split('_').str[1]

    weeks = sorted(df['Anno_Settimana'].unique(), reverse=True)

    # 2. Dynamic Column Naming
    kpis = list(trend_config.keys())
    status_cols = [f'Status_{kpi}_{window}w' for kpi in kpis if f'Status_{kpi}_{window}w' in df.columns]

    with pd.ExcelWriter(output_filename, engine='xlsxwriter', engine_kwargs={'options': {'nan_inf_to_errors': True}}) as writer:
        workbook = writer.book

        # --- Styles ---
        header_fmt = workbook.add_format({'bold': True, 'bg_color': '#D7E4BC', 'border': 1, 'align': 'center'})
        num_fmt    = workbook.add_format({'num_format': '0.00', 'border': 1})
        pct_fmt    = workbook.add_format({'num_format': '0.0"%"', 'border': 1, 'align': 'center'}) # Format for Percentiles
        flag_fmt   = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'border': 1, 'align': 'center'})
        normal_fmt = workbook.add_format({'border': 1, 'align': 'center'})

        # NUOVO STILE: per l'allineamento a sinistra delle prime due colonne
        left_fmt   = workbook.add_format({'border': 1, 'align': 'left'})

        title_fmt  = workbook.add_format({'bold': True, 'font_size': 14, 'font_color': '#2F5597'})
        route_header_fmt = workbook.add_format({'bold': True, 'bg_color': '#D9E1F2', 'border': 1})

        for week in weeks:
            df_week = df[df['Anno_Settimana'] == week].copy()
            sheet_name = str(week)

            # --- SUMMARY SECTION ---
            df_week['Total_Flags'] = df_week[status_cols].sum(axis=1)
            # Indici: 0=Short_Viste, 1=Relazione, 2=Total_Flags, 3+ =status_cols
            summary_cols = ['Short_Viste', 'Relazione', 'Total_Flags'] + status_cols
            # Stable sort: First by flags (descending), then alphabetically to break ties
            summary_df = df_week[summary_cols].sort_values(['Total_Flags', 'Short_Viste'], ascending=[False, True])

            worksheet = workbook.add_worksheet(sheet_name)
            writer.sheets[sheet_name] = worksheet
            worksheet.write(0, 0, f"WEEKLY EXECUTIVE SUMMARY: {week} ({window}w Trend)", title_fmt)

            for c_idx, col in enumerate(summary_df.columns):
                worksheet.write(2, c_idx, col, header_fmt)

            for r_idx, row_vals in enumerate(summary_df.values):
                for c_idx, val in enumerate(row_vals):

                    if c_idx < 2:
                        # Colonne 0 (Short_Viste) e 1 (Relazione): Allineamento a sinistra
                        fmt = left_fmt
                    elif c_idx > 2 and val == 1:
                        # Colonne status (dall'indice 3 in poi) = 1: Sfondo rosso
                        fmt = flag_fmt
                    else:
                        # Colonna 2 (Total_Flags) o status = 0/NaN: Formato normale centrato
                        fmt = normal_fmt

                    worksheet.write(r_idx + 3, c_idx, "" if pd.isna(val) else val, fmt)

            # (Opzionale) Puoi anche allargare leggermente le prime due colonne per mostrare meglio l'allineamento
            worksheet.set_column(1, 1, 15)

            # --- ROUTE DETAIL SECTION ---
            current_row = len(summary_df) + 7
            for short_name in summary_df['Short_Viste']:
                viste_full = df_week[df_week['Short_Viste'] == short_name]['Viste'].iloc[0]
                route_all = df[df['Viste'] == viste_full]

                # We need ALL history up to this week for the percentile math
                full_history = route_all[route_all['Anno_Settimana'] <= week]

                # We slice the tail for the specific rolling/W-1 visual columns
                window_history = full_history.tail(window + 1)
                if window_history.empty: continue

                curr_row_data = window_history.iloc[-1]
                prev_history = window_history.iloc[:-1].iloc[::-1]

                detail_rows = []
                for kpi in kpis:
                    if kpi not in curr_row_data: continue

                    s_col = f'Status_{kpi}_{window}w'
                    roll_col = f'RollingAvg_{kpi}_{window}w'

                    roll_val = curr_row_data[roll_col] if roll_col in curr_row_data else np.nan
                    status_val = curr_row_data[s_col] if s_col in curr_row_data else 0
                    threshold_val = trend_config[kpi][1]

                    # --- PERCENTILE LOGIC ---
                    kpi_all_past = full_history[kpi].dropna().iloc[:-1] # Drop nulls and exclude current week
                    k_weeks = len(kpi_all_past)

                    if k_weeks > 0 and not pd.isna(curr_row_data[kpi]):
                        # Calculates what % of past weeks were <= this week's value
                        pct_rank = (kpi_all_past <= curr_row_data[kpi]).mean() * 100
                    else:
                        pct_rank = np.nan

                    # --- BUILD ROW ---
                    row = {
                        'KPI': kpi,
                        'Status': status_val,
                        'Settimana corrente': curr_row_data[kpi],
                        f'Media {window} set. preced.': roll_val,
                        'Threshold (+/-)': threshold_val,
                        'Percentile': pct_rank,
                        'Storico (set.)': k_weeks
                    }

                    for i, (idx, p_w) in enumerate(prev_history.iterrows()):
                        row[f'W-{i+1}'] = p_w[kpi] if kpi in p_w else np.nan

                    detail_rows.append(row)

                if not detail_rows: continue

                detail_df = pd.DataFrame(detail_rows)
                worksheet.write(current_row - 1, 0, f"Dettaglio Relazione: {viste_full}", route_header_fmt)

                for c_idx, col in enumerate(detail_df.columns):
                    worksheet.write(current_row, c_idx, col, header_fmt)

                for r_idx, r_vals in enumerate(detail_df.values):
                    for c_idx, val in enumerate(r_vals):
                        col_name = detail_df.columns[c_idx]

                        # Dynamic formatting based on column
                        if col_name == 'Status' and val == 1:
                            fmt = flag_fmt
                        elif col_name == 'Percentile' and not pd.isna(val):
                            fmt = pct_fmt
                        elif col_name == 'Storico (set.)':
                            fmt = normal_fmt # Integer doesn't need decimal places
                        elif isinstance(val, (int, float)) and col_name != 'Status':
                            fmt = num_fmt
                        else:
                            fmt = normal_fmt

                        worksheet.write(current_row + r_idx + 1, c_idx, "" if pd.isna(val) else val, fmt)

                current_row += len(detail_df) + 4

    print(f"📄 Report '{output_filename}' successfully generated with Percentiles.")

## Wrapper functions

### Wrapper 1

In [15]:
def data_cruncher(zip_file_path, folder_path='/content/raw_data'):

  # extracting data from zip and storing it into a folder path
  folder_path=extract_uploaded_data(zip_file_path, extract_to_folder=folder_path)

  # DF creation
  df_punctuality, df_suppressions=process_raw_network_data(folder_path)

  return df_punctuality, df_suppressions

### Wrapper 2

In [16]:
def clean_aggregate_filter(df, rules_dict, group_cols, avg_trains_column, columns_to_keep=None, routes_to_drop=None, min_trains_per_day=3 ):

  # Sanity checks on the df
  clean_df=run_sanity_checks(df, rules_dict, columns_to_keep)

  # Add a check for empty DataFrame after sanity checks
  if clean_df.empty:
    print("⚠️ Input DataFrame is empty after sanity checks. Skipping aggregation and filtering.")
    return clean_df # Return the empty DataFrame

  # Aggregate data by week
  df_weekly=aggregate_data_weekly(clean_df, group_cols)

  # Filter out rows with sketchy looking values
  df_filtered=filter_weekly(df_weekly, avg_trains_column, routes_to_drop, min_trains_per_day)

  if avg_trains_column == 'Media Treni Circolati':
    df_final=pivot_and_group_directions(df_filtered)
  elif avg_trains_column == 'Media_Programmati_Supp':
    df_final=df_filtered
  else:
    print("WTH is going on? it looks like its not punctuality or suppressions")
    df_final=df_filtered
  return df_final

### Wrapper 3

In [17]:
def KPIs_dataframe(df_punct_final=None, df_supp_final=None, merge_type='inner'):

  # Create master dataset
  df_master=create_master_dataset(df_punct_final, df_supp_final, merge_type=merge_type)

  # Calculate the final metrics
  df_metrics=calculate_final_metrics(df_master)

  return df_metrics

### Wrapper 4

In [18]:
def final_report(df_master_base, trend_config, TARGET_WINDOW):

  # 1. Calculate Historical Trends
  df_test = df_master_base.copy()
  processed_df = calculate_historical_trends(
      df=df_test,
      metrics_config=trend_config,
      windows=[TARGET_WINDOW] # You can calculate multiple, but we focus the report on one
  )

  # 2. Generate the Report
  generate_executive_report(
      df=processed_df,
      trend_config=trend_config,
      window=TARGET_WINDOW,
      output_filename=f'Executive_Punctuality_Report_{TARGET_WINDOW}weeks_2.xlsx'
  )

## Test of the wrapper functions


**In the following cells is the operative script, which might become a function if I feel like it!**\
It will contain **3** **Sections**, delineated with the 3 wrapper functions



1.   The **Data import** and **Categorization** into Punctuality and Suppression df (function *data_cruncher*)
2.   The **Sanity checks**, **Aggregation** into weekly data, **Filtration** of non indicative routes and weeks (function *clean_aggregate_filter*)\
This will run twice for both Punctuality ans Suppression df
3.   The **Merging**, **Calculation** of the Final Metrics, and Computation of the **Rolling Averages**, Deltas and Status of the week (function *final_dataframe*)

The script will **contain** all the necessary **Variables** that will be separated into their respective sections.

---
First we define the variables in the following cell, then we run the functions.


In [19]:
### ------------------------------------------
### WRAPPER 2 VARIABLES

### ------------------------------------------
###  DEFINING YOUR SPECIFIC RULES (rules_dict)

tol = 0.01
# --- rules for punctuality ---
punctuality_rules = {
    # 1. Uniqueness Check
    "Duplicate Rows (Route, Date, Dir)": lambda df: df.duplicated(subset=['Short_Viste', 'Giorno', 'Direction'], keep='first'),

    # 2. Integrity Checks
    "Media != Circolati": lambda df: df['Media Treni Circolati'].round(2) != df['Circolati'],
    "Circolati != In + Fuori Fascia": lambda df: df['Circolati'] != (df['In Fascia'] + df['Fuori Fascia']),
    "Fuori Fascia != Sum of categories": lambda df: df['Fuori Fascia'] != df[['Fuori Fascia Per Esterne', 'Fuori Fascia per RFI', 'Fuori Fascia per Propria IF', 'Fuori Fascia per Altre IF']].sum(axis=1),
    "RFI != Sum of RFI causes": lambda df: df['Fuori Fascia per RFI'] != df[['RFI Attr. Automatica', 'RFI Circ', 'RFI Impianti e Interaz TT', 'RFI Lavori e IF di RFI', 'RFI Incov di Eserc', 'RFI Indotte']].sum(axis=1),

    # 3. Math Validity Checks (Used for strict data dropping)
    "Invalid Punt. Reale": lambda df: ~(np.isclose(df['Punt. Reale'], (df['In Fascia'] / df['Circolati'].replace(0, np.nan)) * 100, atol=tol) | (df['Circolati'] == 0)),
    "Invalid Punt. SB": lambda df: ~(np.isclose(df['Punt. SB'], ((df['In Fascia'] + df['Fuori Fascia Per Esterne'] + df['Fuori Fascia per Altre IF']) / df['Circolati'].replace(0, np.nan)) * 100, atol=tol) | (df['Circolati'] == 0)),
    "Invalid Punt. RFI": lambda df: ~(np.isclose(df['Punt. RFI'], ((df['Circolati'] - df['Fuori Fascia per RFI']) / df['Circolati'].replace(0, np.nan)) * 100, atol=tol) | (df['Circolati'] == 0)),
    "Invalid Punt. IF": lambda df: ~(np.isclose(df['Punt. IF'], ((df['Circolati'] - df['Fuori Fascia per Propria IF']) / df['Circolati'].replace(0, np.nan)) * 100, atol=tol) | (df['Circolati'] == 0)),
    "Invalid Punt. B1d": lambda df: ~(np.isclose(df['Punt. B1d'], ((df['In Fascia'] + df['Fuori Fascia Per Esterne']) / df['Circolati'].replace(0, np.nan)) * 100, atol=tol) | (df['Circolati'] == 0))
}

# --- rules for suppressions ---
suppression_rules = {
    "Duplicate Rows (Route, Date)": lambda df: df.duplicated(subset=['Short_Viste', 'Giorno'], keep='first'),
    "Traccia Programmata Mismatch (P vs T)": lambda df: df['Treni con Traccia Programmata_P'].notna() & (df['Treni con Traccia Programmata_P'] != df['Treni con Traccia Programmata_T']),
    "Circolati Mismatch (P vs T)": lambda df: df['Treni Circolati_P'].notna() & (df['Treni Circolati_P'] != df['Treni Circolati_T']),
    "Prog - Circ != Soppressi + Non Rilevati": lambda df: (df['Treni con Traccia Programmata_T'] - df['Treni Circolati_T']) != (df['Treni Soppressi_T'] + df['Treni non Rilevati_T']),
    "Negative Partial Suppressions": lambda df: df['Treni Soppressi_P'] < 0,
    "Negative Total Suppressions": lambda df: df['Treni Soppressi_T'] < 0,
    "More Suppressions Than Programmed": lambda df: (df['Treni Soppressi_T'].fillna(0) + df['Treni Soppressi_P'].fillna(0)) > df['Treni con Traccia Programmata_T']
}



### ------------------------------------------
### DEFINING COLUMNS TO AGGREGATE

# group_cols (PUNCTUALITY)
punct_groups = ['Short_Viste', 'Viste', 'Anno_Settimana', 'Direction']

# group_cols (SUPPRESSIONS)
supp_groups = ['Short_Viste', 'VISTE', 'Anno_Settimana']




### ------------------------------------------
### DEFINE 'AVG NUMBER OF TRAINS' COLUMN

# avg_trains_column (PUNCTUALITY)
avg_trains_column_punct = 'Media Treni Circolati'

# avg_trains_column (SUPPRESSIONS)
avg_trains_column_supp = 'Media_Programmati_Supp'



### ------------------------------------------
### DEFINING COLUMNS TO KEEP IN THE DF

# columns_to_keep (PUNCTUALITY)
punct_cols_to_keep = [
    'Short_Viste', 'Viste', 'Anno_Settimana', 'Giorno',
    'Media Treni Circolati', 'Circolati', 'In Fascia', 'Fuori Fascia',
    'Fuori Fascia per RFI', 'Fuori Fascia per Propria IF', 'Fuori Fascia per Altre IF',
    'Direction'
]
# columns_to_keep (SUPPRESSIONS)
supp_cols_to_keep = [
    'Short_Viste', 'VISTE', 'Anno_Settimana', 'Giorno',
    'Treni con Traccia Programmata_T', 'Treni Circolati_T', 'Treni Soppressi_T',
    'Gestore Infrastruttura Treni_T', 'Impresa Ferroviaria Treni_T',
    'Media_Programmati_Supp', 'Media_Circolati_Supp', 'Treni Soppressi_P',
    'Gestore Infrastruttura Treni_P', 'Impresa Ferroviaria Treni_P'
]

### ------------------------------------------
### ROUTES TO DROP

routes_we_exclude = ['23. MI1', '24. MI2', '25. MI2', '26. MI1']

### ------------------------------------------
### WRAPPER 4 VARIABLES

# Define Variables Configuration
trend_config = {
    'Punt. Reale': ('lower_is_bad', 6.0),
    'Punt. RFI': ('lower_is_bad', 2.0),
    'Punt. IF': ('lower_is_bad', 2.0),
    'Tasso_Soppressioni_Totali': ('higher_is_bad', 1.0),
    'Tasso_Soppressioni_Parziali': ('higher_is_bad', 1.0),
    'Diff_True_Punct': ('either_is_bad', 8.0),
    'Diff_Circolati_Norm': ('either_is_bad', 2.0)
}

TARGET_WINDOW = 4



In [20]:
import time
print("\n" + "█"*60)
print("🚀 INITIATING NETWORK PERFORMANCE PIPELINE 🚀")
print("█"*60 + "\n")

### SECTION 1. DATA IMPORT and CATEGORIZATION

# OUTPUTS
# df_punct : DATA-FRAME that contains all of the punctuality rows
# df_supp  : DATA-FRAME that contains all of the suppression rows

# INPUTS
# 1 - zip_file_path : the path to the zipped file
# 2 - folder_path   : the path that will contain the two DFs (Assigned Automatically to '/content/raw_data' if not otherwise specified)


print("="*60)
print("🛠️  PHASE 1: DATA IMPORT & CATEGORIZATION")
print("="*60)
# 1. Record the start time
start_time = time.perf_counter()

df_punct, df_supp = data_cruncher('/content/paridis2425.zip', '/content/raw_data6')

# 2. Record the end time
end_time = time.perf_counter()
# 3. Calculate the difference
execution_time = end_time - start_time
print(f"The code took for Phase 1 {execution_time:.4f} seconds to run.")

### SECTION 2. CLEAN AGGREGATE and FILTER
# It takes one df (either suppressions or punctualities) and it removes the

# OUTPUTS
# df_final : DATA-FRAME cleaned from anomalies, aggregated by week, grouped by direction.

# INPUTS
# 1 - df              : The df that we want to clean
# 2 - rules_dict      : A dictionary containing the rules that you want to enforce in the df, to make sure no anomalies are found
# 3 - group_cols      : The columns(keys) used to aggregate the daily data into weekly (route, week)
# 4 - avg_trains_column : The name of the column in the df that contains the avg number of trains (used to filter extremely low traffic weeks)
# 5 - columns_to_keep : A list of the columns to keep in the df to not oversaturate it
# 6 - routes_to_drop  : A list of the routes that we want to exclude from the study
# 7 - min_trains_per_day : The number of daily trains we accept as minimum threshold

print("\n" + "="*60)
print("📈  PHASE 2: WEEKLY AGGREGATION & CLEANING")
print("="*60)
# 1. Record the start time
start_time = time.perf_counter()

# Now the function will run for the punctuality df
df_final_punct=clean_aggregate_filter(df_punct, punctuality_rules, punct_groups, avg_trains_column_punct, punct_cols_to_keep, routes_we_exclude, min_trains_per_day=3)

# Now the function will run for the suppression df
df_final_supp=clean_aggregate_filter(df_supp, suppression_rules, supp_groups, avg_trains_column_supp, supp_cols_to_keep, routes_we_exclude, min_trains_per_day=3)

# 2. Record the end time
end_time = time.perf_counter()
# 3. Calculate the difference
execution_time = end_time - start_time
print(f"The code took for Phase 2 {execution_time:.4f} seconds to run.")

### SECTION 3. MERGE, METRICS, AND TRENDS
# Takes the finalized punctuality and suppression dataframes, stitches them together,
# calculates the final network KPIs

# OUTPUTS
# df_master_trends : DATA-FRAME containing the fully merged weekly data, final percentages,

# INPUTS
# 1 - df_punct_final : The cleaned, weekly, pivoted punctuality dataframe
# 2 - df_supp_final  : The cleaned, weekly suppression dataframe
# 3 - merge_type     : How to stitch them together ('inner', 'left', 'outer')


print("\n" + "="*60)
print("🧠  PHASE 3: MASTER MERGE")
print("="*60)
# 1. Record the start time
start_time = time.perf_counter()
# Add a check here to ensure there is data before calling KPIs_dataframe
if df_final_punct.empty and df_final_supp.empty:
    print("❌ No data available after cleaning and aggregation. Skipping Master Merge and KPI calculation.")
    df_master_base = pd.DataFrame() # Ensure df_master_base is an empty DataFrame to prevent downstream errors
else:
    df_master_base = KPIs_dataframe(df_final_punct, df_final_supp, merge_type='outer')

# 2. Record the end time
end_time = time.perf_counter()
# 3. Calculate the difference
execution_time = end_time - start_time
print(f"The code took for Phase 3 {execution_time:.4f} seconds to run.")

print("\n" + "⭐"*30)
if df_master_base.empty:
    print(f"❌ PHASE 3 FAILED! Master Dataset is empty. No metrics can be calculated due to no input data.")
else:
    print(f"✅ PHASE 3 SUCCESSFUL! Master Dataset rows: {len(df_master_base)}")
print("⭐"*30 + "\n")

### SECTION 4. TREND CALCULATION AND REPORT GENERATION
# Calculates the historical trends based on the Master Dataset and generates
# the final multi-tabbed Excel executive report.

# OUTPUTS
# Executive_Punctuality_Report_{TARGET_WINDOW}weeks.xlsx : The final formatted Excel report.

# INPUTS
# 1 - df_master_base : The fully merged and calculated dataframe from Phase 3
# 2 - trend_config   : Dictionary defining the metrics, evaluation rules, and thresholds
# 3 - TARGET_WINDOW  : The rolling window timeframe (e.g., 4 for a 4-week trend)

print("\n" + "="*60)
print("📊  PHASE 4: TREND CALCULATION & REPORT GENERATION")
print("="*60)
# 1. Record the start time
start_time = time.perf_counter()

# Ensure there is data before attempting to generate the report
if 'df_master_base' not in locals() or df_master_base.empty:
    print("❌ No Master Dataset available. Skipping Report Generation.")
else:
    print(f"⚙️  Generating report for a {TARGET_WINDOW}-week rolling window...")
    # Call the reporting function
    final_report(df_master_base, trend_config, TARGET_WINDOW)

# 2. Record the end time
end_time = time.perf_counter()
# 3. Calculate the difference
execution_time = end_time - start_time
print(f"The code took for Phase 4 {execution_time:.4f} seconds to run.")

print("\n" + "█"*60)
print("🏁 PIPELINE EXECUTION COMPLETED 🏁")
print("█"*60 + "\n")


████████████████████████████████████████████████████████████
🚀 INITIATING NETWORK PERFORMANCE PIPELINE 🚀
████████████████████████████████████████████████████████████

🛠️  PHASE 1: DATA IMPORT & CATEGORIZATION
📦 Extracting /content/paridis2425.zip...
✅ Extraction complete. All files are flattened directly into: /content/raw_data6
📂 Scanning folder: /content/raw_data6


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

🔗 Merging Total and Partial Suppressions...
⚠️ Inner Join Dropped 4452 rows:
   - 4338 rows had Total suppressions but NO Partial suppressions.
   - 114 rows had Partial suppressions but NO Total suppressions.
✅ Done! Ready: Punctuality rows: 202875, Suppressions rows: 281817
The code took for Phase 1 132.9274 seconds to run.

📈  PHASE 2: WEEKLY AGGREGATION & CLEANING
🕵️ Running Sanity Checks (Initial rows: 202875)
                             Rule  Failed Rows
Duplicate Rows (Route, Date, Dir)       135250
               Media != Circolati            0
   Circolati != In + Fuori Fascia            0
Fuori Fascia != Sum of categories            0
         RFI != Sum of RFI causes           12
              Invalid Punt. Reale            0
                 Invalid Punt. SB            0
                Invalid Punt. RFI            0
                 Invalid Punt. IF            0
                Invalid Punt. B1d           12
🧹 Purged 135258 total invalid rows.
✂️ Trimmed down to 12 core c